# 03 - Matched edit budget: does targeting the surprising words matter?

**The question.** In the last run SIRA changed only about 15% of the words, and a plain paraphrase changed 68%, so they could not be compared fairly. This run compares SIRA with the **random** and **lowest-surprisal** controls at the **same share of changed words**, with the quality gates switched to *measure only* (they no longer cancel the attack).

**Everything is already set up in this notebook. You do not need to paste or edit anything.** It re-uses the texts and the paraphrase results from your last run (folder `sira_runs/quick2` on Google Drive), so only the new attacks are computed.

**Before you start:** set **Runtime -> Change runtime type -> T4 GPU**, and switch on notebook access for `HF_TOKEN` (key icon, left sidebar). Then run the cells from top to bottom, one at a time.

## 1. Log in

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
print("HF user:", whoami()["name"])

## 2. Get the code (always a fresh copy)
If Colab asks to restart the session, do it, then rerun cells 1 and 2.

In [ ]:
!rm -rf /content/sira-lab
!git clone -q https://github.com/deymonp/sira-lab.git /content/sira-lab
%cd /content/sira-lab
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

## 3. Settings
Colab will ask you to allow access to Google Drive. Allow it. The last two lines should say the attacks list and `True`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN = "/content/drive/MyDrive/sira_runs/quick2"     # same folder as the last run: its texts and paraphrase results are re-used

import os
from sira_lab.config import load_config
from sira_lab.lab.run import run_generate, run_attack, run_evaluate
cfg = load_config("configs/lab_dose.yaml")
print("attacks, in order of priority:", cfg.lab.attacks)
have = os.path.exists(f"{RUN}/generated.jsonl")
print("texts from the last run found:", have)
if not have:
    print("not found, generating them again (this takes a while)")
    run_generate(cfg, RUN)

## 4. Stage 2: the new attacks
This is the long stage. The attacks run in priority order: the first two (`sira@0.3`, `random@0.3`) answer the main question, the others refine it.

* The cell prints the time it took at the end. Please keep that number.
* **You can stop it at any time** with the square stop button on the left of the cell. Nothing is lost. Run the cell again later and it continues where it stopped.
* If Colab disconnects: rerun cells 1, 2 and 3, then this cell.

In [ ]:
import time
t0 = time.time()
run_attack(cfg, RUN)
print(f"Stage 2 took {(time.time() - t0) / 60:.1f} minutes")

## 5. Stage 3: score everything
Run this when Stage 2 has finished, or after you have stopped it: attacks that did not finish are skipped and listed as `no results yet`. It loads each detector in turn, so expect a few minutes.

In [ ]:
import time, pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
t0 = time.time()
df = run_evaluate(cfg, RUN)
print(f"Stage 3 took {(time.time() - t0) / 60:.1f} minutes")
display(df[["detector", "algo", "attack", "detect_rate", "rate_lo", "rate_hi", "changed_mean", "sim_median", "mean_score"]].round(3))
display(Image(f"{RUN}/dose_response.png"))

## 6. Head-to-head comparison (no GPU needed)
For each detector, SIRA and its control are compared **text by text** on the same texts. `mean_diff` below zero means SIRA hides the text better; `ci_lo` / `ci_hi` is the 95% range. If the range includes zero, the verdict is `no clear difference`.

In [ ]:
from sira_lab.lab.analysis import load_scores, paired_table
pt = paired_table(load_scores(RUN))
print(pt.round(3).to_string())

## 7. What to send me
Your results are saved in the Drive folder `My Drive / sira_runs / quick2`. Open it in your browser (drive.google.com), download these four files and attach them in the chat:

`results.csv`, `scores.csv`, `paired.csv`, `dose_response.png`

Also tell me the two times printed under Stage 2 and Stage 3. When you are done, click **Runtime -> Disconnect and delete runtime**.

In [ ]:
!ls -la {RUN}